# Capitolo 13 — Correttezza (Python)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap13_correttezza_python.ipynb)

Questo notebook accompagna il capitolo 13 «Correttezza» delle dispense (volume 4, *Analisi e ricorsione*) e della [pagina del sito](https://fabiofurini.github.io/algoritmi/analisi/13-correttezza/). Contiene il codice **Python** degli algoritmi con pseudocodice del capitolo: `FindMaximum`, `InsertionSort`, `BinarySearch`, `EuclideanAlgorithm`. Per ogni algoritmo il codice **stampa l'invariante di ciclo a ogni valutazione della condizione** (prima del ciclo e dopo ogni iterazione) e controlla che sia vera: le tabelle sono quelle del capitolo. Il controllo di un esempio non dimostra la correttezza (la si dimostra con inizializzazione, mantenimento e terminazione), ma mostra l'invariante all'opera.

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime ▸ Esegui tutto* (oppure cella per cella con Maiusc+Invio). Non serve installare nulla.

Ogni algoritmo è una funzione Python; le celle di esecuzione lo lanciano sugli stessi esempi delle dispense e stampano i risultati.

**Indici.** Nello pseudocodice gli array sono indicizzati da 1; nel codice Python da 0: l'elemento `a[i]` dello pseudocodice è `a[i-1]` nel codice (e un ciclo `while i <= n` da 1 diventa `while i < n` da 0). Le tabelle stampate usano gli indici dello pseudocodice.

## 1. Ricerca del massimo: FindMaximum

**Input:** un array $a[1..n]$ di $n \ge 1$ numeri interi. **Output:** il massimo di $a$. **Invariante:** ogni volta che si valuta la condizione, $max = \max\{a[1],\dots,a[i-1]\}$.

Pseudocodice: sezione 13.2 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi/13-correttezza/).

In [1]:
def print_invariant(a, i, max_val):
    """Invariante: max = massimo di a[1..i-1]."""
    sub = a[:i]                                    # a[1..i-1] dello pseudocodice
    ok = (max_val == max(sub))
    print("%2d | %3d | %-26s | %14d | %s" % (i + 1, max_val, sub, max(sub), "ok" if ok else "FALSA"))


def find_maximum(a):
    n = len(a)
    max_val = a[0]
    i = 1                                          # i := 2
    print_invariant(a, i, max_val)                 # prima della prima valutazione della condizione
    while i < n:                                   # i <= n
        if a[i] > max_val:
            max_val = a[i]
        i = i + 1
        print_invariant(a, i, max_val)             # dopo ogni iterazione, cioe' prima della valutazione successiva
    return max_val

In [2]:
a = [9, 12, 3, 16, 19, 11, 12]      # esempio delle dispense
print("a = %s, n = %d" % (a, len(a)))
print(" i | max | a[1..i-1]                  | max{a[1..i-1]} | invariante")
m = find_maximum(a)
print("Il ciclo termina con i = n + 1 = %d: FindMaximum restituisce %d" % (len(a) + 1, m))

a = [9, 12, 3, 16, 19, 11, 12], n = 7
 i | max | a[1..i-1]                  | max{a[1..i-1]} | invariante
 2 |   9 | [9]                        |              9 | ok
 3 |  12 | [9, 12]                    |             12 | ok
 4 |  12 | [9, 12, 3]                 |             12 | ok
 5 |  16 | [9, 12, 3, 16]             |             16 | ok
 6 |  19 | [9, 12, 3, 16, 19]         |             19 | ok
 7 |  19 | [9, 12, 3, 16, 19, 11]     |             19 | ok
 8 |  19 | [9, 12, 3, 16, 19, 11, 12] |             19 | ok
Il ciclo termina con i = n + 1 = 8: FindMaximum restituisce 19


**Prova tu.** Cambia l'array: l'invariante resta vero a ogni riga.

In [3]:
a = [4, 8, 8, 1, 10, 2]      # <-- cambia l'array
print(" i | max | a[1..i-1]                  | max{a[1..i-1]} | invariante")
m = find_maximum(a)
print("restituisce %d" % m)

 i | max | a[1..i-1]                  | max{a[1..i-1]} | invariante
 2 |   4 | [4]                        |              4 | ok
 3 |   8 | [4, 8]                     |              8 | ok
 4 |   8 | [4, 8, 8]                  |              8 | ok
 5 |   8 | [4, 8, 8, 1]               |              8 | ok
 6 |  10 | [4, 8, 8, 1, 10]           |             10 | ok
 7 |  10 | [4, 8, 8, 1, 10, 2]        |             10 | ok
restituisce 10


## 2. Ordinamento per inserimento: InsertionSort

**Input:** un array $a[1..n]$ di $n \ge 1$ numeri naturali. **Output:** l'array ordinato in modo non decrescente. **Invariante esterno:** $a[1..i-1]$ contiene gli elementi originali di $a[1..i-1]$ ordinati e $a[i..n]$ è uguale all'originale. **Invariante interno** (I1)–(I3): $a[1..j] = b[1..j]$, $a[j+2..i] = b[j+1..i-1]$ e questi elementi sono $> key$.

Pseudocodice: sezione 13.3 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi/13-correttezza/).

In [4]:
def split(a, i):
    """L'array con una barra dopo la parte ordinata a[1..i-1]."""
    return "[" + ", ".join(str(x) for x in a[:i]) + " | " + ", ".join(str(x) for x in a[i:]) + "]"


def print_outer(a, orig, i):
    """Invariante esterno: a[1..i-1] = elementi originali di a[1..i-1] ordinati, a[i..n] = originale."""
    ok = a[:i] == sorted(orig[:i]) and a[i:] == orig[i:]
    print("%2d | %-26s | %-19s | %s" % (i + 1, split(a, i), orig[:i], "ok" if ok else "FALSA"))


def print_inner(a, b, i, j, key):
    """Invariante interno (I1)-(I3); b = a a inizio iterazione."""
    ip, jp = i + 1, j + 1                          # indici dello pseudocodice
    left, right = a[:jp], a[jp + 1:ip]             # a[1..j] e a[j+2..i]
    ok = (left == b[:jp] and right == b[jp:ip - 1] and all(x > key for x in right))
    if j >= 0:
        cond = "a[%d] = %d %s %d: %s" % (jp, a[j], ">" if a[j] > key else "<=", key, "true" if a[j] > key else "false")
    else:
        cond = "j = 0: false"
    print("%d | %-15s | %-12s | %-9s | %-20s | %s" % (jp, a, left, right, cond, "ok" if ok else "FALSA"))


def insertion_sort(a, show_outer=True, trace_inner_at=0):
    n = len(a)
    orig = a[:]                                    # copia dell'input: serve solo a verificare l'invariante
    i = 1                                          # i := 2
    if show_outer:
        print_outer(a, orig, i)
    while i < n:                                   # i <= n
        b = a[:]                                   # contenuto di a a inizio iterazione (solo per l'invariante interno)
        key = a[i]
        j = i - 1
        tr = (i + 1 == trace_inner_at)
        if tr:
            print_inner(a, b, i, j, key)
        while j >= 0 and a[j] > key:               # j >= 1 and a[j] > key
            a[j + 1] = a[j]
            j = j - 1
            if tr:
                print_inner(a, b, i, j, key)
        a[j + 1] = key
        i = i + 1
        if show_outer:
            print_outer(a, orig, i)

In [5]:
# Il ciclo esterno: a[1..5] = (5, 2, 4, 1, 3), esempio delle dispense
a = [5, 2, 4, 1, 3]
print("a = %s" % a)
print(" i | a (ordinato | resto)       | a[1..i-1] originale | invariante")
insertion_sort(a)
print("ordinato: %s" % a)

# Il ciclo interno all'iterazione i = 5, con key = 3 (a = (1, 2, 4, 5, 3) all'inizio)
print()
a = [5, 2, 4, 1, 3]
print("j | a               | a[1..j]      | a[j+2..i] | condizione           | invariante")
insertion_sort(a, False, 5)
print("dopo la riga 10 (a[j+1] = key): a = %s" % a)

a = [5, 2, 4, 1, 3]
 i | a (ordinato | resto)       | a[1..i-1] originale | invariante
 2 | [5 | 2, 4, 1, 3]           | [5]                 | ok
 3 | [2, 5 | 4, 1, 3]           | [5, 2]              | ok
 4 | [2, 4, 5 | 1, 3]           | [5, 2, 4]           | ok
 5 | [1, 2, 4, 5 | 3]           | [5, 2, 4, 1]        | ok
 6 | [1, 2, 3, 4, 5 | ]         | [5, 2, 4, 1, 3]     | ok
ordinato: [1, 2, 3, 4, 5]

j | a               | a[1..j]      | a[j+2..i] | condizione           | invariante
4 | [1, 2, 4, 5, 3] | [1, 2, 4, 5] | []        | a[4] = 5 > 3: true   | ok
3 | [1, 2, 4, 5, 5] | [1, 2, 4]    | [5]       | a[3] = 4 > 3: true   | ok
2 | [1, 2, 4, 4, 5] | [1, 2]       | [4, 5]    | a[2] = 2 <= 3: false | ok
dopo la riga 10 (a[j+1] = key): a = [1, 2, 3, 4, 5]


**Prova tu.** Cambia l'array: ogni riga della tabella esterna deve restare `ok`.

In [6]:
a = [6, 3, 8, 1, 9, 2]      # <-- cambia l'array
print(" i | a (ordinato | resto)       | a[1..i-1] originale | invariante")
insertion_sort(a)

 i | a (ordinato | resto)       | a[1..i-1] originale | invariante
 2 | [6 | 3, 8, 1, 9, 2]        | [6]                 | ok
 3 | [3, 6 | 8, 1, 9, 2]        | [6, 3]              | ok
 4 | [3, 6, 8 | 1, 9, 2]        | [6, 3, 8]           | ok
 5 | [1, 3, 6, 8 | 9, 2]        | [6, 3, 8, 1]        | ok
 6 | [1, 3, 6, 8, 9 | 2]        | [6, 3, 8, 1, 9]     | ok
 7 | [1, 2, 3, 6, 8, 9 | ]      | [6, 3, 8, 1, 9, 2]  | ok


## 3. Ricerca binaria: BinarySearch

**Input:** un array $a[1..n]$ ordinato in modo non decrescente e un valore $m$. **Output:** un indice $k$ con $a[k] = m$ se $m$ è in $a$, $-1$ altrimenti. **Invariante:** (I1) $1 \le left$ e $right \le n$; (I2) se $m$ è in $a$, allora $m$ è in $a[left..right]$.

Pseudocodice: sezione 13.4 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi/13-correttezza/).

In [7]:
def print_row(a, m, left, right, mid):
    """Invariante (I1) 1 <= left e right <= n, (I2) se m e' in a allora e' in a[left..right]; mid = -1: intervallo vuoto."""
    sub = a[left:right + 1]
    ok = (left + 1 >= 1 and right + 1 <= len(a) and (m not in a or m in sub))
    if mid < 0:
        action = "left > right => return -1"
    elif a[mid] == m:
        action = "mid = %d, a[%d] = %d = %d => return %d" % (mid + 1, mid + 1, a[mid], m, mid + 1)
    elif a[mid] < m:
        action = "mid = %d, a[%d] = %d < %d => left = %d" % (mid + 1, mid + 1, a[mid], m, mid + 2)
    else:
        action = "mid = %d, a[%d] = %d > %d => right = %d" % (mid + 1, mid + 1, a[mid], m, mid)
    print("%4d | %5d | %-27s | %-11s | %s" % (left + 1, right + 1, sub, "ok" if ok else "FALSA", action))


def binary_search(a, m, trace=False):
    n = len(a)
    left = 0                                       # left := 1
    right = n - 1                                  # right := n
    while left <= right:
        mid = left + (right - left) // 2
        if trace:
            print_row(a, m, left, right, mid)      # riga dell'invariante a questa valutazione
        if a[mid] == m:
            return mid
        if a[mid] < m:
            left = mid + 1
        else:
            right = mid - 1
    if trace:
        print_row(a, m, left, right, -1)
    return -1

In [8]:
a = [3, 9, 11, 12, 12, 16, 19]      # esempio delle dispense
print("a = %s, n = %d" % (a, len(a)))

m = 10
print()
print("m = %d" % m)
print("left | right | a[left..right]              | invariante  | azione")
k = binary_search(a, m, True)
print("m = %d non trovato: restituisce %d" % (m, k))

m = 12
print()
print("m = %d" % m)
print("left | right | a[left..right]              | invariante  | azione")
k = binary_search(a, m, True)
print("m = %d trovato: indice %d nel codice (posizione %d nello pseudocodice)" % (m, k, k + 1))

# Controllo esaustivo per m = 0, ..., 20
errors = 0
for m in range(0, 21):
    k = binary_search(a, m)
    if (m in a and (k < 0 or a[k] != m)) or (m not in a and k != -1):
        errors += 1
print()
print("Controllo per m = 0, ..., 20: %d errori" % errors)

a = [3, 9, 11, 12, 12, 16, 19], n = 7

m = 10
left | right | a[left..right]              | invariante  | azione
   1 |     7 | [3, 9, 11, 12, 12, 16, 19]  | ok          | mid = 4, a[4] = 12 > 10 => right = 3
   1 |     3 | [3, 9, 11]                  | ok          | mid = 2, a[2] = 9 < 10 => left = 3
   3 |     3 | [11]                        | ok          | mid = 3, a[3] = 11 > 10 => right = 2
   3 |     2 | []                          | ok          | left > right => return -1
m = 10 non trovato: restituisce -1

m = 12
left | right | a[left..right]              | invariante  | azione
   1 |     7 | [3, 9, 11, 12, 12, 16, 19]  | ok          | mid = 4, a[4] = 12 = 12 => return 4
m = 12 trovato: indice 3 nel codice (posizione 4 nello pseudocodice)

Controllo per m = 0, ..., 20: 0 errori


**Prova tu.** Cambia l'array (deve restare ordinato) e il valore cercato.

In [9]:
a = [2, 4, 4, 7, 9, 15, 21, 30]      # <-- cambia l'array (ordinato!)
m = 15                               # <-- e il valore cercato
print("left | right | a[left..right]              | invariante  | azione")
k = binary_search(a, m, True)
print("restituisce %d" % k)

left | right | a[left..right]              | invariante  | azione
   1 |     8 | [2, 4, 4, 7, 9, 15, 21, 30] | ok          | mid = 4, a[4] = 7 < 15 => left = 5
   5 |     8 | [9, 15, 21, 30]             | ok          | mid = 6, a[6] = 15 = 15 => return 6
restituisce 5


## 4. Algoritmo di Euclide: EuclideanAlgorithm

**Input:** due numeri naturali $n, m \ge 1$. **Output:** $\gcd(n, m)$. **Invariante:** $n \ge 1$, $m \ge 0$ e $\gcd(n,m) = \gcd(n_0,m_0)$, dove $n_0, m_0$ sono i valori in input.

Pseudocodice: sezione 13.5 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi/13-correttezza/).

In [10]:
def gcd_check(n, m):                               # mcd per enumerazione dei divisori, indipendente dall'algoritmo
    if m == 0:
        return n
    d = min(n, m)
    while n % d != 0 or m % d != 0:
        d = d - 1
    return d


def print_row(n, m, g0):
    """Invariante: n >= 1, m >= 0 e gcd(n, m) = gcd(n0, m0)."""
    g = gcd_check(n, m)
    ok = (n >= 1 and m >= 0 and g == g0)
    print("%4d | %4d | %9d | %s" % (n, m, g, "ok" if ok else "FALSA"))


def euclidean_algorithm(n, m):
    g0 = gcd_check(n, m)                           # gcd(n0, m0), solo per verificare l'invariante
    print_row(n, m, g0)
    while m != 0:
        r = n % m
        n = m
        m = r
        print_row(n, m, g0)
    return n

In [11]:
n, m = 252, 105      # esempio delle dispense
print("(n0, m0) = (%d, %d)" % (n, m))
print("   n |    m | gcd(n, m) | invariante")
g = euclidean_algorithm(n, m)
print("Il ciclo termina con m = 0: restituisce n = %d" % g)

print()
n, m = 48, 18
print("(n0, m0) = (%d, %d)" % (n, m))
print("   n |    m | gcd(n, m) | invariante")
g = euclidean_algorithm(n, m)
print("Il ciclo termina con m = 0: restituisce n = %d" % g)

(n0, m0) = (252, 105)
   n |    m | gcd(n, m) | invariante
 252 |  105 |        21 | ok
 105 |   42 |        21 | ok
  42 |   21 |        21 | ok
  21 |    0 |        21 | ok
Il ciclo termina con m = 0: restituisce n = 21

(n0, m0) = (48, 18)
   n |    m | gcd(n, m) | invariante
  48 |   18 |         6 | ok
  18 |   12 |         6 | ok
  12 |    6 |         6 | ok
   6 |    0 |         6 | ok
Il ciclo termina con m = 0: restituisce n = 6


**Prova tu.** Cambia i due numeri: il $\gcd$ resta costante a ogni riga.

In [12]:
n, m = 1071, 462      # <-- cambia i due numeri
print("   n |    m | gcd(n, m) | invariante")
g = euclidean_algorithm(n, m)
print("restituisce %d" % g)

   n |    m | gcd(n, m) | invariante
1071 |  462 |        21 | ok
 462 |  147 |        21 | ok
 147 |   21 |        21 | ok
  21 |    0 |        21 | ok
restituisce 21
